# 新浪财经研报抓取与盈利预测提取

从新浪财经研报列表页抓取指定股票的全部研究报告，进入每篇详情页提取正文，
并从正文中解析出「年份 → 预测值」的盈利预测数据（归母净利润、EPS、PE、营收等）。

输出 CSV 列：标题 / 发布日期 / 抓取日期 / 机构 / 研究员 / 分析结果(json) / URL

In [1]:
import re
import json
import time
import html as html_lib
from datetime import date

import requests
import pandas as pd

SYMBOL = '300910'  # 股票代码，改这里即可抓其他股票
LIST_URL = ('https://stock.finance.sina.com.cn/stock/go.php/vReport_List/kind/search/'
            'index.phtml?t1=2&symbol={symbol}&p={page}')

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0 Safari/537.36',
}
TODAY = date.today().isoformat()

session = requests.Session()
session.headers.update(HEADERS)

def detail_url(path):
    """列表页 href 以 // 开头（协议相对地址），补上 https: 前缀"""
    return 'https:' + path if path.startswith('//') else path

## 1. 抓取列表页（含分页）

页面是 GB2312 编码，用 `resp.encoding = 'gb18030'` 保证中文不乱码。
分页通过 `&p=N` 参数控制，从分页栏的页码链接中自动发现总页数。

In [2]:
ROW_RE = re.compile(
    r'<tr>\s*<td>\d+</td>\s*<td class="tal f14">\s*'
    r'<a[^>]*href="(?P<url>[^"]+)"[^>]*>\s*(?P<title>[^<]+?)\s*</a>\s*</td>\s*'
    r'<td>(?P<kind>[^<]*)</td>\s*'
    r'<td>(?P<pubdate>\d{4}-\d{2}-\d{2})</td>\s*'
    r'<td>\s*<a[^>]*>\s*<div class="fname05"><span>(?P<org>[^<]*)</span></div>\s*</a>\s*</td>\s*'
    r'<td><div class="fname"><span>(?P<analyst>[^<]*)</span></div></td>',
    re.S)

def fetch_list_page(page):
    resp = session.get(LIST_URL.format(symbol=SYMBOL, page=page), timeout=30)
    resp.encoding = 'gb18030'
    return resp.text

def count_pages(html_text):
    nums = [int(n) for n in re.findall(r"set_page_num\('(\d+)'", html_text)]
    return max(nums) if nums else 1

first_page = fetch_list_page(1)
total_pages = count_pages(first_page)
print(f'共 {total_pages} 页')

reports = []
for page in range(1, total_pages + 1):
    html_text = first_page if page == 1 else fetch_list_page(page)
    rows = ROW_RE.findall(html_text)
    if not rows:
        break
    reports.extend(rows)
    print(f'第 {page}/{total_pages} 页: {len(rows)} 篇')
    time.sleep(2)  # 请求过快会触发新浪 456 限流

print(f'列表页共抓到 {len(reports)} 篇研报')
reports[:2]

共 4 页
第 1/4 页: 35 篇


第 2/4 页: 39 篇


第 3/4 页: 36 篇


第 4/4 页: 27 篇


列表页共抓到 137 篇研报


[('//stock.finance.sina.com.cn/stock/go.php/vReport_Show/kind/search/rptid/838033462402/index.phtml',
  '瑞丰新材(300910)：润滑油添加剂26Q2出口稳健 看好公司增长持续',
  '创业板',
  '2026-07-22',
  '西部证券股份有限公司',
  '李旋坤/王金源'),
 ('//stock.finance.sina.com.cn/stock/go.php/vReport_Show/kind/search/rptid/836314487209/index.phtml',
  '瑞丰新材(300910)：润滑油添加剂龙头 稳健成长进击全球',
  '创业板',
  '2026-07-02',
  '东方证券股份有限公司',
  '陈传双')]

## 2. 抓取详情页正文

In [3]:
BODY_RE = re.compile(r'<div class="blk_container">(.*?)</div>', re.S)

def fetch_report_text(path, retries=3):
    """path 形如 //stock.finance.sina.com.cn/.../rptid/838033462402/index.phtml
    新浪对高频请求返回 HTTP 456 限流，遇到时等待后重试"""
    url = detail_url(path)
    for attempt in range(retries):
        resp = session.get(url, timeout=30)
        if resp.status_code == 200:
            break
        wait = 60 * (attempt + 1)
        print(f'  限流({resp.status_code})，等 {wait}s 重试')
        time.sleep(wait)
    resp.encoding = 'gb18030'
    m = BODY_RE.search(resp.text)
    if not m:
        return ''
    text = re.sub(r'<br\s*/?>', '\n', m.group(1))
    text = re.sub(r'<[^>]+>', '', text)
    text = html_lib.unescape(text)
    text = text.replace('　', ' ').replace('\xa0', ' ')
    return re.sub(r'[ \t]+', ' ', text).strip()

# 测试一篇
sample = fetch_report_text(reports[0][0])
print(reports[0][1])
print(sample[-400:])

瑞丰新材(300910)：润滑油添加剂26Q2出口稳健 看好公司增长持续
降8.4%，环比增长15.6%，出口均价为2.03 万元/吨，其中6 月出口量同比增长15.4%、环比增长33.8%，二季度末发货动能已经明显改善。

 
 公司作为国内头部润滑油添加剂公司，量增以及产品结构的升级是可持续的。我们认为，公司后续核心看点除了出口的增长持续之外，还有高端复合剂持续放量、API 及OEM 认证突破、国际核心客户拓展，以及海外项目推动后的中东、非洲等市场本地化供应。行业出口市场正由传统目的地向新兴市场分散。

 
 投资建议：公司是国内的润滑油添加剂的头部企业，受益于行业景气度的持续。我们预计公司26-28 年归母净利分别为7.89/9.47/11.43 亿元，对应PE分别为13.8/11.5/9.5X，维持“增持”评级。

 
 风险提示：全球润滑油需求不及预期、海外市场竞争加剧、原材料价格及汇率波动、高端产品认证进度不及预期、海外建设及客户导入不及预期。


## 3. 从正文解析盈利预测

研报中的预测写法多样，常见的有：
- 「我们预计公司26-28年归母净利分别为7.89/9.47/11.43亿元」
- 「预计公司2022-2024年归母净利润分别为2.6亿元、3.4亿元及4.9亿元」
- 「我们预测26-28年公司EPS为2.22/2.58/2.99元」

解析策略：只看含「预计/预测/预期」的句子 → 找年份区间（支持 26-28 与 2026-2028 两种写法）→
在该句各分句中找指标名 + 连续数值串，数值个数与年份数一致才采纳。

In [4]:
FORECAST_HINT = re.compile(r'预计|预测|预期|有望|调整|修正|下调|上调|维持')
# 年份三种写法:
#   区间式  "26-28年" / "2024-2026年" / "2026年-2028年" (斜杠也算: "2026年/2027年")
#   枚举式  "22/23年"
#   组合式  "2025-2026年、新增2027年"
YEAR_RANGE = re.compile(r'(20\d{2}|\d{2})\s*年?\s*[-–—~至/]\s*(20\d{2}|\d{2})\s*年')
YEAR_LIST = re.compile(r'((?:20\d{2}|\d{2})\s*年(?:\s*[/、,，]\s*(?:20\d{2}|\d{2})\s*年)+)')
YEAR_PLUS = re.compile(
    r'((?:20\d{2}|\d{2})\s*年?\s*[-–—~至]\s*(?:20\d{2}|\d{2})\s*年(?:\s*[、,，]?\s*(?:新增|及|和)\s*(?:20\d{2}|\d{2})\s*年)+)')
NUM_PAT = r'[+-]?\d+(?:\.\d+)?'  # 带符号，支持增速写法 +25.47%
UNIT_PAT = r'(?:亿元|万元|亿|元|X|倍|%)'
SEP = r'[/、,，及和\\\\]+'  # 分隔符含反斜杠，如 18.3\14.7\13.4倍
VALUES = re.compile(NUM_PAT + r'(?:\s*(?:' + UNIT_PAT + r')?\s*' + SEP + r'\s*' + NUM_PAT + r')*')
METRICS = [(r'归母净利', '归母净利润'), (r'净利润', '净利润'), (r'每股收益|EPS|盈利预测|盈利', 'EPS'),
           (r'营业收入|营收', '营业收入'), (r'PE', 'PE'), (r'增速|增长率|增长', '增速')]

def norm_year(y):
    y = int(y)
    return y + 2000 if y < 100 else y

def collect_year_spans(sentence):
    """返回 [(start, [年份...]), ...]，三种写法各自识别后按位置去重"""
    spans = []
    for m in YEAR_PLUS.finditer(sentence):
        ys = sorted(set(norm_year(y) for y in re.findall(r'20\d{2}|\d{2}', m.group(1))))
        if 1 < len(ys) <= 7:
            spans.append((m.start(), [str(y) for y in ys]))
    for m in YEAR_LIST.finditer(sentence):
        ys = sorted(set(norm_year(y) for y in re.findall(r'20\d{2}|\d{2}', m.group(1))))
        if 1 < len(ys) <= 7 and ys[-1] - ys[0] <= len(ys) + 1:
            spans.append((m.start(), [str(y) for y in ys]))
    for m in YEAR_RANGE.finditer(sentence):
        y1, y2 = norm_year(m.group(1)), norm_year(m.group(2))
        if 0 <= y2 - y1 <= 6:
            spans.append((m.start(), [str(y) for y in range(y1, y2 + 1)]))
    spans.sort(key=lambda sp: (sp[0], -len(sp[1])))
    dedup = []
    for sp in spans:
        if dedup and sp[0] < dedup[-1][0] + 12:
            if len(sp[1]) > len(dedup[-1][1]):
                dedup[-1] = sp
        else:
            dedup.append(sp)
    return dedup

def pick_years(spans, abs_pos, n):
    before = [sp for sp in spans if sp[0] < abs_pos]
    for sp in reversed(before):
        if len(sp[1]) == n:
            return sp[1]
    for sp in reversed(before):
        lo = min(int(y) for y in sp[1])
        hi = max(int(y) for y in sp[1])
        if lo + n - 1 >= hi and n <= 7:
            return [str(y) for y in range(lo, lo + n)]
    return None

def parse_forecast(text):
    """返回 {指标名(单位): {年份: 数值}}，如 {'归母净利润(亿元)': {'2026': 7.89, ...}}"""
    text = re.sub(r'（[^）]*）|\([^)]*\)', ' ', text)  # 去括号内容，避免混入"前值"
    result = {}
    for sentence in re.split(r'[。\n]', text):
        if not FORECAST_HINT.search(sentence):
            continue
        spans = collect_year_spans(sentence)
        if not spans:
            continue
        for m in re.finditer(r'[^，,；;]+', sentence):
            clause = m.group(0)
            base = m.start()
            candidates = []
            for pat, name in METRICS:
                for mm in re.finditer(pat, clause):
                    candidates.append((mm.start(), mm.end(), name))
            candidates.sort(key=lambda c: (-(c[1] - c[0]), -c[0]))
            taken = []
            for cstart, cend, name in candidates:
                if any(cstart < te and cend > ts for ts, te, _ in taken):
                    continue
                vm = VALUES.search(clause, cend)
                if vm is None:
                    continue
                vals = [float(v) for v in re.findall(NUM_PAT, vm.group(0))]
                if not vals:
                    continue
                if any(vm.start() == vs for _, _, vs in taken):
                    continue
                gap_text = clause[cend:vm.start()]
                if len(re.sub(r'[\s的为分别至元\dkxX%．.+/、,，和及()]', '', gap_text)) > 6:
                    continue
                years = pick_years(spans, base + vm.start(), len(vals))
                if years is None:
                    continue
                tail = clause[vm.end(): vm.end() + 4]
                um = re.match(r'\s*(' + UNIT_PAT + r')', tail)
                unit = um.group(1) if um else ''
                key = f'{name}({unit})' if unit else name
                result.setdefault(key, {}).update(dict(zip(years, vals)))
                taken.append((cstart, cend, vm.start()))
    return result

# 自测几种真实写法
for s in [
    '我们预计公司2026 年/2027 年EPS 分别为2.15 元/2.54 元，2025-2027 年CAGR 为14.9%',
    '考虑到公司Q4 归母净利润超预期，上调2025-2026 年、新增2027 年EPS 分别为2.89、3.48、4.09 元（原为2.77、3.33、-元）',
    '投资建议：考虑到公司放量迅速，上调2021-2022 年盈利预测至3.5亿和4.5 亿',
    '维持 2021-2022 年 EPS，新增 2023 年 EPS，2021-2023 年 EPS 分别为 2.44、3.29、4.44 元',
    '我们修正公司2021-2023 年公司归母净利润分别为3.3、4.8 和5.9 亿元；EPS 分别为2.2、3.2 和3.9 元',
    '我们预计公司26-28 年归母净利分别为7.89/9.47/11.43 亿元，对应PE分别为13.8/11.5/9.5X',
    '2016-2025 年营收、净利润九年复合增速分别达30.6%、40.7%',
]:
    print(json.dumps(parse_forecast(s), ensure_ascii=False), '<=', s[:35])


{"EPS(元)": {"2026": 2.15, "2027": 2.54}} <= 我们预计公司2026 年/2027 年EPS 分别为2.15 元/2.
{"EPS(元)": {"2025": 2.89, "2026": 3.48, "2027": 4.09}} <= 考虑到公司Q4 归母净利润超预期，上调2025-2026 年、新增20
{"EPS(亿)": {"2021": 3.5, "2022": 4.5}} <= 投资建议：考虑到公司放量迅速，上调2021-2022 年盈利预测至3.
{"EPS(元)": {"2021": 2.44, "2022": 3.29, "2023": 4.44}} <= 维持 2021-2022 年 EPS，新增 2023 年 EPS，20
{"归母净利润(亿元)": {"2021": 3.3, "2022": 4.8, "2023": 5.9}, "EPS(元)": {"2021": 2.2, "2022": 3.2, "2023": 3.9}} <= 我们修正公司2021-2023 年公司归母净利润分别为3.3、4.8 
{"归母净利润(亿元)": {"2026": 7.89, "2027": 9.47, "2028": 11.43}, "PE(X)": {"2026": 13.8, "2027": 11.5, "2028": 9.5}} <= 我们预计公司26-28 年归母净利分别为7.89/9.47/11.43
{} <= 2016-2025 年营收、净利润九年复合增速分别达30.6%、40.


## 4. 全量抓取并生成 CSV

In [5]:
records = []
for i, (url, title, kind, pubdate, org, analyst) in enumerate(reports, 1):
    try:
        text = fetch_report_text(url)
        forecast = parse_forecast(text)
    except Exception as e:
        print(f'[{i}/{len(reports)}] 失败 {title[:30]}: {e}')
        forecast = {}
    records.append({
        '标题': title.strip(),
        '发布日期': pubdate,
        '抓取日期': TODAY,
        '机构': org.strip(),
        '研究员': analyst.strip(),
        '分析结果': json.dumps(forecast, ensure_ascii=False),
        'URL': detail_url(url),
    })
    if i % 10 == 0:
        print(f'[{i}/{len(reports)}] 已完成')
    time.sleep(3)

df = pd.DataFrame(records, columns=['标题', '发布日期', '抓取日期', '机构', '研究员', '分析结果', 'URL'])
from pathlib import Path
OUT_DIR = Path.cwd() / 'output'
OUT_DIR.mkdir(exist_ok=True)
output_csv = OUT_DIR / f'sina_{SYMBOL}_forecast.csv'
df.to_csv(output_csv, index=False, encoding='utf-8-sig')
print(f'已保存 {len(df)} 行 -> {output_csv}')
df.head(10)


[10/137] 已完成


[20/137] 已完成


[30/137] 已完成


[40/137] 已完成


[50/137] 已完成


...

[110/137] 已完成


[120/137] 已完成


[130/137] 已完成


已保存 137 行 -> D:\codeproject\binwh.quant-pilot\notebooks\data\output\sina_300910_forecast.csv


,标题,发布日期,抓取日期,机构,研究员,分析结果,URL
0,瑞丰新材(300910)：润滑油添加剂26Q2出口稳健 看好公司增长持续,2026-07-22,2026-08-19,西部证券股份有限公司,李旋坤/王金源,"{""归母净利润(亿元)"": {""2026"": 7.89, ""2027"": 9.47, ""20...",https://stock.finance.sina.com.cn/stock/go.php...
1,瑞丰新材(300910)：润滑油添加剂龙头 稳健成长进击全球,2026-07-02,2026-08-19,东方证券股份有限公司,陈传双,"{""EPS(元)"": {""2026"": 2.22, ""2027"": 2.58, ""2028""...",https://stock.finance.sina.com.cn/stock/go.php...
2,瑞丰新材(300910)：产销放量产能持续扩张 红海布局加速海外成长,2026-06-20,2026-08-19,国投证券股份有限公司,王华炳,"{""增速(%)"": {""2026"": 25.47, ""2027"": 15.34, ""2028...",https://stock.finance.sina.com.cn/stock/go.php...
3,瑞丰新材(300910)：国产龙头崛起 迈向全球润滑油添加剂核心舞台,2026-06-10,2026-08-19,中国国际金融股份有限公司,徐啸天/傅锴铭/裘孝锋,"{""EPS(元)"": {""2026"": 2.15, ""2027"": 2.54}}",https://stock.finance.sina.com.cn/stock/go.php...
4,瑞丰新材(300910)：润滑油添加剂产销稳健增长 费用端增...,2026-05-05,2026-08-19,长江证券股份有限公司,马太/李禹默,"{""归母净利润(亿元)"": {""2026"": 9.1, ""2027"": 11.0, ""202...",https://stock.finance.sina.com.cn/stock/go.php...
5,瑞丰新材(300910)：产品毛利率维持稳定 加快全球化布局,2026-04-30,2026-08-19,华安证券股份有限公司,王强峰,"{""归母净利润(亿元)"": {""2026"": 9.1, ""2027"": 10.86, ""20...",https://stock.finance.sina.com.cn/stock/go.php...
6,瑞丰新材(300910)：业绩符合预期 产品结构调整叠加费用...,2026-04-25,2026-08-19,上海申银万国证券研究所有限公司,李绍程/宋涛/马昕晔,"{""归母净利润(亿元)"": {""2026"": 9.03, ""2027"": 11.46, ""2...",https://stock.finance.sina.com.cn/stock/go.php...
7,瑞丰新材(300910)：润滑油添加剂产销稳健增长 产品矩阵加速升级,2026-03-30,2026-08-19,华安证券股份有限公司,王强峰,"{""归母净利润(亿元)"": {""2026"": 9.1, ""2027"": 10.86, ""20...",https://stock.finance.sina.com.cn/stock/go.php...
8,瑞丰新材(300910)：添加剂产销量进一步增长 新项目建设...,2026-03-24,2026-08-19,东方财富证券股份有限公司,张志扬/梅宇鑫,"{""营业收入(亿元)"": {""2026"": 43.34, ""2027"": 51.92, ""2...",https://stock.finance.sina.com.cn/stock/go.php...
9,瑞丰新材(300910)点评：业绩基本符合预期 全年销量再创...,2026-03-23,2026-08-19,上海申银万国证券研究所有限公司,李绍程/宋涛/马昕晔,"{""归母净利润(亿元)"": {""2026"": 10.94, ""2027"": 13.19}, ...",https://stock.finance.sina.com.cn/stock/go.php...


## 5. 结果检查

统计提取成功率，抽查解析出的 JSON。

In [6]:
hit = df['分析结果'] != '{}'
print(f'提取到预测的研报: {hit.sum()} / {len(df)} ({hit.mean():.0%})')

# 抽查最新几篇的解析结果
for _, row in df[hit].head(5).iterrows():
    print(f"\n[{row['发布日期']}] {row['标题'][:40]}")
    print(json.dumps(json.loads(row['分析结果']), ensure_ascii=False, indent=2))

提取到预测的研报: 136 / 137 (99%)

[2026-07-22] 瑞丰新材(300910)：润滑油添加剂26Q2出口稳健 看好公司增长持续
{
  "归母净利润(亿元)": {
    "2026": 7.89,
    "2027": 9.47,
    "2028": 11.43
  },
  "PE(X)": {
    "2026": 13.8,
    "2027": 11.5,
    "2028": 9.5
  }
}

[2026-07-02] 瑞丰新材(300910)：润滑油添加剂龙头 稳健成长进击全球
{
  "EPS(元)": {
    "2026": 2.22,
    "2027": 2.58,
    "2028": 2.99
  }
}

[2026-06-20] 瑞丰新材(300910)：产销放量产能持续扩张 红海布局加速海外成长
{
  "增速(%)": {
    "2026": 25.47,
    "2027": 15.34,
    "2028": 14.97
  },
  "净利润(%)": {
    "2026": 18.51,
    "2027": 19.99,
    "2028": 18.3
  },
  "归母净利润(%)": {
    "2026": 18.81,
    "2027": 19.99,
    "2028": 18.3
  }
}

[2026-06-10] 瑞丰新材(300910)：国产龙头崛起 迈向全球润滑油添加剂核心舞台
{
  "EPS(元)": {
    "2026": 2.15,
    "2027": 2.54
  }
}

[2026-05-05] 瑞丰新材(300910)：润滑油添加剂产销稳健增长 费用端增...
{
  "归母净利润(亿元)": {
    "2026": 9.1,
    "2027": 11.0,
    "2028": 13.3
  }
}
